In [5]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Optional (install xgboost if needed)
from xgboost import XGBRegressor

# -----------------------------
# LOAD DATA
# -----------------------------
import pandas as pd

df = pd.read_excel("hourlyLoadDataIndia.xlsx") # your dataset
df.head()
df.info()


# Rename columns
df = df.rename(columns={
    "datetime": "datetime",
    "National Hourly Demand": "national_load",
    "Northen Region Hourly Demand": "north",
    "Western Region Hourly Demand": "west",
    "Eastern Region Hourly Demand": "east",
    "Southern Region Hourly Demand": "south",
    "North-Eastern Region Hourly Demand": "north_east"
})

print(df.columns)
# -----------------------------
# PREPROCESSING
# -----------------------------
df["datetime"] = pd.to_datetime(df["datetime"])

# Drop datetime (already extracted features exist)
df = df.drop(columns=["datetime"])



<class 'pandas.DataFrame'>
RangeIndex: 46728 entries, 0 to 46727
Data columns (total 7 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   datetime                            46728 non-null  datetime64[us]
 1   National Hourly Demand              46728 non-null  float64       
 2   Northen Region Hourly Demand        46728 non-null  float64       
 3   Western Region Hourly Demand        46728 non-null  float64       
 4   Eastern Region Hourly Demand        46728 non-null  float64       
 5   Southern Region Hourly Demand       46728 non-null  float64       
 6   North-Eastern Region Hourly Demand  46728 non-null  float64       
dtypes: datetime64[us](1), float64(6)
memory usage: 2.5 MB
Index(['datetime', 'national_load', 'north', 'west', 'east', 'south',
       'north_east'],
      dtype='str')


In [6]:

# Target
y = df["national_load"]

# Features
X = df.drop(columns=["national_load"])

# -----------------------------
# TRAIN TEST SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# -----------------------------
# METRICS FUNCTION
# -----------------------------
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mae, rmse, r2, mape

# -----------------------------
# MODELS
# -----------------------------
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1)
}

results = []

# -----------------------------
# TRAIN LOOP
# -----------------------------
for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae, rmse, r2, mape = evaluate(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape
    })

    # Save model
    joblib.dump(model, f"{name.replace(' ', '_')}.pkl")

# -----------------------------
# SAVE RESULTS
# -----------------------------
results_df = pd.DataFrame(results)
results_df.to_csv("model_results.csv", index=False)

print("\n✅ Training Complete!")
print(results_df)

Training Linear Regression...
Training Random Forest...
Training XGBoost...

✅ Training Complete!
               Model          MAE         RMSE        R2      MAPE
0  Linear Regression     0.004849     0.007132  1.000000  0.000003
1      Random Forest  2645.552163  4415.616092  0.948843  1.335305
2            XGBoost  1970.348557  3326.687129  0.970964  0.999116
